# 03 – Train IsolationForest

Placeholder: train anomaly model (e.g. IsolationForest) and save to ml/artifacts/.

In [2]:
# =========================
# 03_train_isoforest.ipynb
# Train Isolation Forest on HDFS window frequency features
# =========================

import json
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
import hashlib

from sklearn.ensemble import IsolationForest
import joblib

In [3]:
# ---------- Paths ----------
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "machine_learning").exists() and (REPO_ROOT.parent / "machine_learning").exists():
    REPO_ROOT = REPO_ROOT.parent

ML_DIR = REPO_ROOT / "machine_learning"
DATA_DIR = ML_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"

FEATURES_DIR = ML_DIR / "features"
MODELS_DIR = ML_DIR / "models" / "isoforest"
RESULTS_DIR = ML_DIR / "results" / "isoforest"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

STRUCTURED_PATH = PROCESSED_DIR / "hdfs_structured.parquet"
X_PATH = FEATURES_DIR / "hdfs_window_features.npz"
ENCODER_PATH = FEATURES_DIR / "hdfs_encoder.json"
MODEL_PATH = MODELS_DIR / "isoforest.joblib"

print("REPO_ROOT:", REPO_ROOT)
print("STRUCTURED_PATH:", STRUCTURED_PATH)
print("X_PATH:", X_PATH)
print("ENCODER_PATH:", ENCODER_PATH)
print("MODEL_PATH:", MODEL_PATH)

REPO_ROOT: /Users/jay027/Desktop/github_repo/log-analysis-AI-agent
STRUCTURED_PATH: /Users/jay027/Desktop/github_repo/log-analysis-AI-agent/machine_learning/data/processed/hdfs_structured.parquet
X_PATH: /Users/jay027/Desktop/github_repo/log-analysis-AI-agent/machine_learning/features/hdfs_window_features.npz
ENCODER_PATH: /Users/jay027/Desktop/github_repo/log-analysis-AI-agent/machine_learning/features/hdfs_encoder.json
MODEL_PATH: /Users/jay027/Desktop/github_repo/log-analysis-AI-agent/machine_learning/models/isoforest/isoforest.joblib


In [4]:
# ---------- Load ----------
if not STRUCTURED_PATH.exists():
    raise FileNotFoundError(f"Missing structured parquet: {STRUCTURED_PATH}")
if not X_PATH.exists():
    raise FileNotFoundError(f"Missing features npz: {X_PATH}")
if not ENCODER_PATH.exists():
    raise FileNotFoundError(f"Missing encoder json: {ENCODER_PATH}")

df = pd.read_parquet(STRUCTURED_PATH)

with open(ENCODER_PATH, "r") as f:
    enc = json.load(f)

window_size = int(enc["window_size"])
classes_ = enc["classes_"]  # index -> template_id

X = np.load(X_PATH)["X"]

print("df shape:", df.shape)
print("X shape:", X.shape)
print("window_size:", window_size)
print("num templates:", len(classes_))

FileNotFoundError: Missing features npz: /Users/jay027/Desktop/github_repo/log-analysis-AI-agent/machine_learning/features/hdfs_window_features.npz

In [ ]:
# -----------------------------
# Train model
# -----------------------------
params = dict(
    n_estimators=300,
    contamination=0.02,   # tune later
    random_state=42,
    n_jobs=-1
)

iso = IsolationForest(**params)

t0 = time.time()
iso.fit(X)
train_s = time.time() - t0

print(f"Model trained ✅ in {train_s:.2f}s")

In [ ]:
# -----------------------------
# Save model + metadata
# -----------------------------
joblib.dump(iso, MODEL_PATH)

meta = {
    "model_type": "IsolationForest",
    "trained_at_epoch": time.time(),
    "train_seconds": train_s,
    "window_size": window_size,
    "num_templates": int(X.shape[1]),
    "num_windows": int(X.shape[0]),
    "params": params,
    "encoder_path": str(ENCODER_PATH),
    "features_path": str(X_PATH),
    "model_path": str(MODEL_PATH),
}

META_PATH.write_text(json.dumps(meta, indent=2))

print("Saved ✅")
print("MODEL exists?", MODEL_PATH.exists(), "|", MODEL_PATH)
print("META exists?", META_PATH.exists(), "|", META_PATH)